# 🎭 Emotive TTS — Master Colab Pipeline

**COMP3065 Final Year Project**

> **One-click setup:** This notebook clones the project from GitHub and runs
> entirely on Google Colab. Just click **Runtime → Run all**.

## Workflow
1. **Setup** — Mount Drive, clone repo from GitHub, install deps
2. **Data preparation** (S1) — Download & process EmoV-DB
3. **Training** — System A → B → C (S2-S4)
4. **Inference** — Generate eval stimuli (S5)
5. **Evaluation** — Prosody analysis + SER probe + plots (S6)

**Runtime:** GPU → T4 (Runtime → Change runtime type → T4 GPU)

---
**GitHub repo:** https://github.com/jimmy00415/COMP3067_Project

## 0. Setup

In [ ]:
# ── 0a. Mount Google Drive (for persistent checkpoint storage) ──
from google.colab import drive
drive.mount('/content/drive')

import os

# Persistent storage on Google Drive — survives runtime restarts
DRIVE_PROJECT = '/content/drive/MyDrive/COMP3065_Project'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/data/processed', exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_PROJECT}')

In [ ]:
# ── 0b. Clone project from GitHub ──
REPO_URL = 'https://github.com/jimmy00415/COMP3067_Project.git'
LOCAL_PROJECT = '/content/COMP3067_Project'

if os.path.exists(LOCAL_PROJECT):
    print('Repo already cloned — pulling latest...')
    !cd {LOCAL_PROJECT} && git pull
else:
    !git clone {REPO_URL} {LOCAL_PROJECT}

%cd {LOCAL_PROJECT}
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# ── 0c. Check GPU & runtime ──
!nvidia-smi

import torch
print(f'\nPyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU     : {gpu_name}')
    print(f'VRAM    : {vram_gb:.1f} GB')
    if vram_gb < 10:
        print('⚠️  VRAM < 10 GB — training may OOM. Use smaller batch_size.')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 0d. Install dependencies ──
# Core TTS + audio processing
!pip install -q TTS==0.22.0 speechbrain>=1.0 librosa>=0.10
# ML infra
!pip install -q hydra-core>=1.3 omegaconf>=2.3 mlflow>=2.10
# Scientific
!pip install -q pandas matplotlib seaborn scipy scikit-learn soundfile
# Optional: demo
!pip install -q gradio>=4.0

# Install project in editable mode (makes `from src.xxx import ...` work)
!pip install -q -e .

# Verify installation
try:
    from src.data.utils import EMOTION_MAP, EMOTION_LABELS
    from src.models.prosody_heads import build_prosody_heads
    from src.training.train import _compute_mel_spectrogram
    print(f'✅ Project installed. Emotions: {EMOTION_LABELS}')
except ImportError as e:
    print(f'❌ Import failed: {e}\n   Re-run this cell or restart runtime.')

## 1. Data Preparation (S1)

In [ ]:
# ── 1a. Get EmoV-DB dataset ──
# Check Drive first, then offer download options
EMOVDB_DRIVE = f'{DRIVE_PROJECT}/data/raw/EmoV-DB'
EMOVDB_LOCAL = 'data/raw/EmoV-DB'

os.makedirs('data/raw', exist_ok=True)

if os.path.exists(EMOVDB_DRIVE):
    # Symlink from Drive to avoid re-downloading
    if not os.path.exists(EMOVDB_LOCAL):
        !ln -sf {EMOVDB_DRIVE} {EMOVDB_LOCAL}
    print(f'✅ EmoV-DB found on Drive → symlinked to {EMOVDB_LOCAL}')
elif os.path.exists(EMOVDB_LOCAL):
    print(f'✅ EmoV-DB already at {EMOVDB_LOCAL}')
else:
    print('⚠️  EmoV-DB not found. Choose one option:')
    print(f'  Option 1: Upload to Google Drive at {EMOVDB_DRIVE}')
    print(f'  Option 2: Uncomment the download lines below and re-run this cell')
    # ---- Uncomment ONE of the following ----
    # !pip install -q gdown
    # !gdown --id YOUR_GDRIVE_FILE_ID -O /tmp/emovdb.zip
    # !unzip -q /tmp/emovdb.zip -d data/raw/
    # ---- OR use wget/curl if you have a direct link ----
    # !wget -q -O /tmp/emovdb.zip "https://YOUR_LINK"
    # !unzip -q /tmp/emovdb.zip -d data/raw/

In [ ]:
# ── 1b. Run data preparation pipeline ──
from src.data.prepare import prepare_dataset
import yaml

with open('configs/data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f)

# Override raw_dir for Colab path
data_cfg['dataset']['raw_dir'] = EMOVDB_LOCAL

summary = prepare_dataset(data_cfg)
print('\n📊 Dataset summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

# Persist manifests to Drive
!cp -r data/manifests {DRIVE_PROJECT}/data/ 2>/dev/null || true
print(f'\n✅ Manifests saved to Drive')

In [ ]:
# ── 1c. Run Data QA ──
from src.data.qa import generate_qa_report

generate_qa_report(
    manifest_path='data/manifests/full_manifest.csv',
    output_dir='figures/data_qa',
    report_path='docs/data_qa_report.md',
)
print('✅ QA report: docs/data_qa_report.md')

# Quick preview
from IPython.display import Markdown, display
if os.path.exists('docs/data_qa_report.md'):
    with open('docs/data_qa_report.md') as f:
        display(Markdown(f.read()[:2000] + '\n\n*... (truncated)*'))

## 2. Training: System A (S2) — Domain adaptation

System A fine-tunes pretrained VITS on EmoV-DB **without** emotion labels.
This is the domain-adapted baseline to isolate the conditioning effect (A→B).

In [ ]:
# ── 2a. Train System A ──
import yaml, torch
from src.training.train import train

with open('configs/train_a.yaml', 'r') as f:
    train_a_cfg = yaml.safe_load(f)

# Colab overrides
train_a_cfg['use_cuda'] = True
train_a_cfg.setdefault('training', {})['fp16'] = True

# Restore from Drive checkpoint if available
drive_ckpt = f'{DRIVE_PROJECT}/checkpoints/system_a/best.pth'
if os.path.exists(drive_ckpt):
    print(f'📂 Restoring System A checkpoint from Drive: {drive_ckpt}')
    train_a_cfg['init_from'] = drive_ckpt

print('🚀 Starting System A training...')
if torch.cuda.is_available():
    print(f'   VRAM before: {torch.cuda.memory_allocated()/1e9:.2f} GB')
result_a = train(train_a_cfg)
print('✅ System A training complete:', result_a)

In [ ]:
# ── 2b. Save System A checkpoint to Drive ──
!mkdir -p {DRIVE_PROJECT}/checkpoints/system_a
!cp -r checkpoints/system_a/* {DRIVE_PROJECT}/checkpoints/system_a/ 2>/dev/null
print('✅ System A checkpoint saved to Drive')

## 3. Training: System B (S3) — Emotion embedding

System B = System A + `nn.Embedding(4, 192)` for emotion conditioning.
Initialises from the **System A checkpoint** (warm start).

In [ ]:
# ── 3a. Train System B ──
with open('configs/train_b.yaml', 'r') as f:
    train_b_cfg = yaml.safe_load(f)

train_b_cfg['use_cuda'] = True
train_b_cfg.setdefault('training', {})['fp16'] = True

# Warm-start from System A
drive_ckpt_a = f'{DRIVE_PROJECT}/checkpoints/system_a/best.pth'
local_ckpt_a = 'checkpoints/system_a/best.pth'
if os.path.exists(drive_ckpt_a):
    train_b_cfg['init_from'] = drive_ckpt_a
elif os.path.exists(local_ckpt_a):
    train_b_cfg['init_from'] = local_ckpt_a

print('🚀 Starting System B training...')
result_b = train(train_b_cfg)
print('✅ System B training complete:', result_b)

In [ ]:
# ── 3b. Save System B checkpoint to Drive ──
!mkdir -p {DRIVE_PROJECT}/checkpoints/system_b
!cp -r checkpoints/system_b/* {DRIVE_PROJECT}/checkpoints/system_b/ 2>/dev/null
print('✅ System B checkpoint saved to Drive')

## 4. Training: System C (S4) — Prosody supervision

System C = System B + utterance-level prosody auxiliary heads (F0 + energy).
Initialises from the **System B checkpoint** (warm start). Loss weight λ = 0.1.

In [ ]:
# ── 4a. Train System C ──
with open('configs/train_c.yaml', 'r') as f:
    train_c_cfg = yaml.safe_load(f)

train_c_cfg['use_cuda'] = True
train_c_cfg.setdefault('training', {})['fp16'] = True

# Warm-start from System B
drive_ckpt_b = f'{DRIVE_PROJECT}/checkpoints/system_b/best.pth'
local_ckpt_b = 'checkpoints/system_b/best.pth'
if os.path.exists(drive_ckpt_b):
    train_c_cfg['init_from'] = drive_ckpt_b
elif os.path.exists(local_ckpt_b):
    train_c_cfg['init_from'] = local_ckpt_b

print('🚀 Starting System C training...')
result_c = train(train_c_cfg)
print('✅ System C training complete:', result_c)

In [ ]:
!cp -r checkpoints/system_c {DRIVE_PROJECT}/checkpoints/
print('System C checkpoint saved to Drive')

## 5. Inference (S5)

Generate evaluation stimuli: 16 canary texts × 4 emotions × 4 systems

In [ ]:
from src.inference.run import run_inference
import yaml

with open('configs/infer.yaml', 'r') as f:
    infer_cfg = yaml.safe_load(f)

infer_cfg['inference']['use_cuda'] = True

results = run_inference(infer_cfg)
print('Inference complete:', results)

In [ ]:
# Save eval stimuli to Drive
!cp -r data/processed/eval_stimuli {DRIVE_PROJECT}/data/processed/
print('Eval stimuli saved to Drive')

## 6. Evaluation (S6)

In [ ]:
# 6a. Prosody analysis (PRIMARY metric)
from src.evaluation.prosody import run_prosody_evaluation
import yaml

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

prosody_results = run_prosody_evaluation(eval_cfg)
print('Prosody evaluation complete')

# Show aggregate stats
if 'agg_stats' in prosody_results:
    display(prosody_results['agg_stats'])

In [ ]:
# 6b. SER probe (AUXILIARY metric — label mismatch caveat applies)
from src.evaluation.ser_probe import run_ser_evaluation

ser_results = run_ser_evaluation(eval_cfg)
print(f"\nSER proxy agreement: {ser_results['ser_proxy_agreement']:.2%}")
print(f"Note: This is an AUXILIARY metric only.")
print(f"Excluded {ser_results['n_excluded']} samples with unmapped emotions (disgust).")

In [ ]:
# 6c. Generate all plots
from src.evaluation.plots import generate_all_plots

generate_all_plots(eval_cfg)
print('All plots saved to figures/')

# Display key plots inline
from IPython.display import Image, display
for plot_name in ['f0_by_system_emotion.png', 'prosody_comparison_grid.png']:
    plot_path = f'figures/{plot_name}'
    if os.path.exists(plot_path):
        display(Image(filename=plot_path, width=800))

In [ ]:
# 6d. Generate listening test stimulus pack
from src.evaluation.listening_test import create_stimulus_pack

stim_summary = create_stimulus_pack(
    manifest_path='data/processed/eval_stimuli/eval_manifest.csv',
    output_dir='outputs/listening_test',
)
print('Stimulus pack:', stim_summary)

In [ ]:
# Save all outputs to Drive
!cp -r tables {DRIVE_PROJECT}/
!cp -r figures {DRIVE_PROJECT}/
!cp -r outputs {DRIVE_PROJECT}/
!cp -r docs {DRIVE_PROJECT}/
print('All outputs saved to Drive')

## 7. Summary

| Step | Status |
|------|--------|
| Data prep | ✅ |
| System A training | ✅ |
| System B training | ✅ |
| System C training | ✅ |
| Inference | ✅ |
| Prosody eval | ✅ |
| SER probe | ✅ |
| Plots | ✅ |
| Listening test | ✅ |

All checkpoints and outputs are saved to Google Drive.